In [1]:
import torch
import matplotlib.pyplot as plt
from typing import Tuple 
import os
import matplotlib.pyplot as plt
from skfmm import distance
from torch.utils.data import DataLoader
from torchvision.datasets import OxfordIIITPet
import preprocess_dataset
import losses
from main_network import DecompositionNetwork
import importlib
importlib.reload(losses)
from tqdm import tqdm
from torchsummary import summary

In [2]:
# Hyperparameters:
LEARNING_RATE = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
NUM_EPOCHS = 3
LOAD_MODEL = False
NUM_WORKERS = os.cpu_count()


In [3]:
train_dataset = preprocess_dataset.OxfordIIITPet_Distancefields_train()
test_dataset = preprocess_dataset.OxfordIIITPet_Distancefields_test()

In [8]:
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


In [5]:
def train(loader, model, optimizer, loss_fn, scaler, device):
    model = model.to(device)

    # data is rgb without background taken out
    # target is in first dim the y_mask and second dim is distance field
    for data, targets in (pbar := tqdm(loader)):
        # take out the background by applying GT mask
        masked_data = preprocess_dataset.mask_rgb_imgs(data, targets)
        masked_data = masked_data.float().to(device)
        targets = targets.float().to(device)

        optimizer.zero_grad()


        # forward pass
        with torch.cuda.amp.autocast():
            predictions = model(masked_data)
            loss = loss_fn(
                predictions, 
                mask_y=torch.squeeze(targets[:][0][0]),
                field_y=torch.squeeze(targets[:][0][1]),
                device=device
                )

        # backward pass
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # update tqdm loop
        pbar.set_postfix(loss=float(loss.item()))

In [6]:
model = DecompositionNetwork(batch_size=BATCH_SIZE)
optim = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scaler = torch.cuda.amp.GradScaler()
# summary(model.to(device=DEVICE), (3, 256, 256))


for epoch in range(NUM_EPOCHS):
    train(train_dataloader, model, optim, losses.all_loss_fn, scaler, device=DEVICE)

model.to(device="cpu")
torch.cuda.empty_cache()

100%|██████████| 230/230 [00:25<00:00,  9.17it/s, loss=2.52] 


In [10]:
_, ax = plt.subplots(1,3, figsize=(10,10))
for x,y in test_dataloader:
    x_masked = preprocess_dataset.mask_rgb_imgs(x,y)
    predict= model(x_masked.float())
    print(predict)
    break
    # ax[0].imshow(torch.permute(x_masked[0], (1,2,0)), cmap="gray")
    # ax[1].imshow(torch.squeeze(y[0][0]), cmap="gray")
    # ax[2].imshow(torch.squeeze(y[0][1]), cmap="gray")
    # break

tensor([[ -32.7440, -237.0667,  333.5529,  334.4213],
        [ -27.8071, -210.4758,  297.4152,  297.4548],
        [ -29.6782, -218.4545,  308.1618,  307.8021],
        [ -31.8821, -223.4058,  313.8649,  314.1282],
        [ -28.4018, -218.4935,  309.3201,  308.7290],
        [ -29.1986, -219.7294,  310.9218,  309.8582],
        [ -31.6404, -238.3681,  337.6548,  336.1698],
        [ -32.4902, -228.6891,  320.9130,  321.5696],
        [ -31.1615, -220.5330,  310.1539,  310.3826],
        [ -29.0810, -209.7479,  295.2676,  295.6388],
        [ -29.6637, -223.8970,  316.9662,  315.6905],
        [ -30.4551, -241.3848,  343.0117,  341.3484],
        [ -31.1961, -231.7522,  327.5866,  327.0376],
        [ -29.4315, -216.7354,  305.8698,  305.5331],
        [ -28.2614, -217.1975,  308.4477,  305.9026],
        [ -24.5122, -208.5278,  299.0003,  295.0818]],
       grad_fn=<AddmmBackward0>)
tensor([[ -28.3051, -216.4213,  305.8374,  306.4123],
        [ -31.4342, -236.2314,  333.3131,  333.7